In [1]:
import re
import requests
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd

In [53]:
import requests
from bs4 import BeautifulSoup
import numpy as np
import re
import pandas as pd

# Lists
Title=[]
Brand=[]
Processor=[]
RAM=[]
ROM=[]
Weight=[]
Current_Price=[]
Original_Price=[]
Ratings=[]
Rating_Count=[]
Review_Count=[]

for page in range(1,19):
    url=f"https://www.flipkart.com/search?q=laptops&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&page={page}"
    req_header = {'Content-Type': 'text/html; charset=UTF-8',
                  'User-Agent': 'Chrome/101.0.0.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/119.0',
                  'Accept-Encoding': 'gzip, deflate, br'}
    
    req=requests.get(url,headers=req_header)
    soup=BeautifulSoup(req.text,"html.parser")
    
    for i in soup.find_all("div",class_="tUxRFH"):

        # Product link
        title_tag = soup_1.find("span", class_="VU-ZEz")
        if title_tag:
            raw_title = title_tag.text.strip()
            match = re.search(r'^([A-Za-z]+\s+[A-Za-z]+)', raw_title)
            if match:
                clean_title = match.group(1)
            else:
                clean_title = np.nan
                Title.append(clean_title)
        else:
            Title.append(np.nan)

       

        # ============================
        # RATING
        # ============================
        rating=soup_1.find("div",class_="XQDdHH")
        Ratings.append(rating.text if rating else np.nan)

        # ============================
        # CURRENT PRICE
        # ============================
        current_price=soup_1.find("div",class_="Nx9bqj CxhGGd")
        Current_Price.append(current_price.text if current_price else np.nan)

        # ============================
        # ORIGINAL PRICE
        # ============================
        original_price=soup_1.find("div",class_="yRaY8j A6+E6v")
        Original_Price.append(original_price.text if original_price else np.nan)

        # ============================
        # Ratings & Reviews Split → Only Rating Count & Review Count
        # ============================
        rating_reviews = soup_1.find("span", class_="Wphh3N")
        if rating_reviews:
            rr_text = rating_reviews.text.strip()

            rating_count = re.findall(r'([\d,]+)\s*Ratings', rr_text)
            review_count = re.findall(r'([\d,]+)\s*Reviews', rr_text)

            Rating_Count.append(rating_count[0].replace(",", "") if rating_count else np.nan)
            Review_Count.append(review_count[0].replace(",", "") if review_count else np.nan)
        else:
            Rating_Count.append(np.nan)
            Review_Count.append(np.nan)

        # ============================
        # BRAND
        # ============================
        brand=re.findall(r'^[A-Za-z]+', raw_title)
        Brand.append(brand[0] if brand else np.nan)

        # ============================
        # RAM (remove GB)
        # ============================
        ram=re.findall(r'\((\d+)\s*GB', raw_title)
        RAM.append(ram[0] if ram else np.nan)

        # ============================
        # ROM (remove GB)
        # ============================
        rom=re.findall(r'(\d+)\s*GB\s*SSD', raw_title)
        ROM.append(rom[0] if rom else np.nan)

        # ============================
        # WEIGHT
        # ============================
        weight=re.findall(r'(\d+\.\d+)\s*Kg', raw_title)
        Weight.append(weight[0] if weight else np.nan)

        
        # ============================
        # Processor
        # ============================
        processor=re.findall(r'\b(i[3579]|Celeron|Ryzen\s+\d+|Ultra\s+\d+|Snapdragon|Helio)\b', raw_title)
        Processor.append(processor[0] if processor else np.nan)

In [60]:

df = pd.DataFrame({
    "Title": Title,
    "Brand": Brand,
    "Processor": Processor,
    "RAM(GB)": RAM,
    "ROM(GB)": ROM,
    "Weight": Weight,
    "Current Price": Current_Price,
    "Original Price": Original_Price,
    "Rating": Ratings,
    "Rating Count": Rating_Count,
    "Review Count": Review_Count
})

df

ValueError: All arrays must be of the same length

In [59]:
len(Title)

0

In [58]:
len(RAM)

432

In [44]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 432 entries, 0 to 431
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Title           431 non-null    object
 1   Brand           432 non-null    object
 2   Processor       360 non-null    object
 3   RAM(GB)         432 non-null    object
 4   ROM(GB)         355 non-null    object
 5   Weight          307 non-null    object
 6   Current Price   431 non-null    object
 7   Original Price  428 non-null    object
 8   Rating          431 non-null    object
 9   Rating Count    425 non-null    object
 10  Review Count    425 non-null    object
dtypes: object(11)
memory usage: 37.3+ KB


In [45]:
df.to_csv('EDA_project.csv')
df.to_excel('EDA_project.xlsx')

In [46]:
df["Current Price"] = df["Current Price"].str.replace("₹","").str.replace("," ,"")

df["Original Price"] = df["Original Price"].str.replace("₹","").str.replace("," ,"")

In [47]:
df

,Title,Brand,Processor,RAM(GB),ROM(GB),Weight,Current Price,Original Price,Rating,Rating Count,Review Count
0,,HP,i3,16,512,1.59,39990,50903,4.1,1999,123
1,,HP,NaN,8,512,1.65,36990,50843,4.2,955,56
2,,Acer,i3,8,512,1.45,27990,43000,4.2,5294,392
3,,Acer,Celeron,8,512,NaN,19990,35999,3.8,6566,586
4,,ASUS,Celeron,8,NaN,1.8,14999,32990,4.2,13,0
...,...,...,...,...,...,...,...,...,...,...,...
427,,Lenovo,i5,16,512,NaN,64890,89990,3.9,185,15
428,,DELL,i3,16,512,NaN,40490,52765,4.2,383,29
429,,HP,NaN,4,256,2.5,27990,32999,4,1774,130
430,,Lenovo,i7,16,NaN,2.4,115990,148090,4.4,70,7


In [ ]:
df['Current Price'] = df['Current Price'].fillna("0")
df['Original Price'] = df['Original Price'].fillna("0")

In [ ]:
df["Current Price"] = df["Current Price"].astype("int")
df["Original Price"] = df["Original Price"].astype("int")



In [ ]:
df.info()


In [ ]:
df['RAM(GB)'] = df['RAM(GB)'].fillna("0")
df['ROM(GB)'] = df['ROM(GB)'].fillna("0")


In [ ]:
df["RAM(GB)"] = df["RAM(GB)"].astype("int")
df["ROM(GB)"] = df["ROM(GB)"].astype("int")

In [ ]:
df.info()

In [ ]:
df["Weight"] = df["Weight"].astype("float")

In [ ]:
df.info()

In [ ]:
df = pd.read_excel("EDA_project.xlsx")

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df['Review Count'] = df['Review Count'].fillna("0")

df['Rating Count'] = df['Rating Count'].fillna("0")

In [ ]:
df['Rating Count']=df['Rating Count'].astype('int')
df['Review Count']=df['Review Count'].astype('int')

In [ ]:
df.info()


In [ ]:
df = df.drop(columns=['Unnamed: 0'])

In [ ]:
df['Weight'].mean()

In [ ]:
df['Weight'].median()

In [ ]:
df['Weight'] = df['Weight'].fillna(
    df.groupby('Brand')['Weight'].transform('median')
)


In [ ]:
df


In [ ]:
df.info()

In [ ]:
df[['Brand','Processor']] 

In [ ]:
df['Brand'] == 'Nan'

In [ ]:
df

In [ ]:
df['Title'].unique()

In [48]:
Title = []
title_tag = soup_1.find("span", class_="VU-ZEz")
if title_tag:
    raw_title = title_tag.text.strip()
    clean_title = re.search(r'^(\w+\s+\w+)', raw_title).group(1)
    clean_title = clean_title.replace("-", "").strip()
    Title.append(clean_title)
else:
    Title.append(np.nan)

In [49]:
Title

['Lenovo LOQ']

In [61]:
df=pd.read_excel('EDA_project.xlsx')

In [62]:
df

,Unnamed: 0,Title,Brand,Processor,RAM(GB),ROM(GB),Weight,Current Price,Original Price,Rating,Rating Count,Review Count
0,0,HP Victus,HP,i5,16,512,2.29,66990,87262,4.2,676,35
1,1,HP MSO 2024,HP,i3,16,512,1.59,39990,50903,4.1,1993,122
2,2,Acer Aspire 3,Acer,Celeron,8,512,1.59,22899,35999,3.8,6451,581
3,3,Acer Aspire 3,Acer,i3,8,512,1.45,27990,43000,4.2,5294,392
4,4,Acer Aspire 15,Acer,Ryzen 5,16,512,1.79,35990,59990,4.1,2879,304
...,...,...,...,...,...,...,...,...,...,...,...,...
427,427,"DELL 14 Plus Backlit Keyboard, Fingerprint reader",DELL,Ultra 7,16,512,1.55,112990,116859,4.7,0,0
428,428,ASUS Vivobook Go 15 (2025) with Office 2024 + ...,ASUS,Ryzen 3,8,512,1.63,33110,43990,4.3,1385,92
429,429,Lenovo IdeaPad Slim 3,Lenovo,i3,8,512,1.70,38750,69990,4.1,152,13
430,430,Lenovo LOQ 2025,Lenovo,i7,16,0,2.40,115990,148090,4.4,70,7


In [67]:
df['Title'].unique()

array(['HP Victus', 'HP MSO 2024', 'Acer Aspire 3', 'Acer Aspire 15',
       'Lenovo IdeaPad Slim 1',
       'Samsung Galaxy Book4 Edge Series Copilot AIPC Full Metal Chasis Qualcomm',
       'HP Laptop', 'DELL 14 Plus Next',
       'MOTOROLA Motobook 60 Pro Full Metal OLED AI PC',
       'Samsung Galaxy Book4 Metal', 'Lenovo Chromebook',
       'ASUS Expertbook P1 Highperformance processor',
       'Lenovo IdeaPad Slim 5 WUXGA OLED Copilot+PC Full Metal Body',
       'Lenovo LOQ Essential', 'Samsung Galaxy Book5 AI Metal',
       'DELL 14 Plus Backlit Keyboard, Fingerprint reader',
       'Lenovo LOQ 2025', 'Acer Aspire 7', 'Acer Nitro V',
       'ASUS Expertbook P1', 'Acer Aspire 3 Backlit',
       "DELL Inspiron 15 MSO'24 with Backlit KB",
       'Acer 3 Years Warranty Aspire 3 (2025)', 'Lenovo 100e Chromebook',
       'ASUS Vivobook 15, with Backlit Keyboard,', 'ASUS Chromebook CX1',
       'Acer Aspire Lite', 'HP', 'ASUS Vivobook Go 15', 'DELL 15',
       'MSI Thin A15', 'ASUS Viv

In [68]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 432 entries, 0 to 431
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      432 non-null    int64  
 1   Title           432 non-null    object 
 2   Brand           432 non-null    object 
 3   Processor       388 non-null    object 
 4   RAM(GB)         432 non-null    int64  
 5   ROM(GB)         432 non-null    int64  
 6   Weight          432 non-null    float64
 7   Current Price   432 non-null    int64  
 8   Original Price  432 non-null    int64  
 9   Rating          432 non-null    float64
 10  Rating Count    432 non-null    int64  
 11  Review Count    432 non-null    int64  
dtypes: float64(2), int64(7), object(3)
memory usage: 40.6+ KB
